In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import dataclasses
import pickle
import os

In [ ]:
import sys, os

GIT_USER  = "JoeyTurn"
GIT_REPO  = "kernels_can_icl"

!git clone https://github.com/{GIT_USER}/{GIT_REPO}.git
!git clone https://github.com/dkarkada/expt-core.git

sys.path.insert(0, "/content/kernels_can_icl")
sys.path.insert(0, "/content/expt-core")

DATASETPATH = "/content/drive/MyDrive/datasets"
RESULTPATH  = "/content/drive/MyDrive/results"
os.environ["DATASETPATH"] = DATASETPATH
os.environ["RESULTPATH"]  = RESULTPATH
os.makedirs(DATASETPATH, exist_ok=True)
os.makedirs(RESULTPATH,  exist_ok=True)


In [ ]:
from misc import rcsetup
rcsetup()

In [ ]:
folder_path = '/content/drive/My Drive/picl/results/kernels/'
file_path_1 = os.path.join(folder_path, 'gaussiankernel_T1.pkl')
with open(file_path_1, 'rb') as f:
        loaded_data_1 = pickle.load(f)
file_path_4 = os.path.join(folder_path, 'gaussiankernel_T4.pkl')
with open(file_path_4, 'rb') as f:
        loaded_data_4 = pickle.load(f)
file_path_16 = os.path.join(folder_path, 'gaussiankernel_T16.pkl')
with open(file_path_16, 'rb') as f:
        loaded_data_16 = pickle.load(f)
file_path_256 = os.path.join(folder_path, 'gaussiankernel_T256.pkl')
with open(file_path_256, 'rb') as f:
        loaded_data_256 = pickle.load(f)

In [ ]:
"""
Plotting helpers for kernel-ICL learning-curve sweeps.
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


# ---------------------------------------------------------------- utilities

def lighten(color, amount=0.5):
    """Interpolate a color toward white (amount>0) or black (amount<0)."""
    rgb = mcolors.to_rgb(mcolors.XKCD_COLORS.get(color, color))
    if amount >= 0:
        new = tuple(c + (1 - c) * amount for c in rgb)
    else:
        new = tuple(c * (1 + amount) for c in rgb)
    return tuple(min(max(x, 0), 1) for x in new)


def ramp(n, base="xkcd:cerulean", lo=-0.15, hi=0.55):
    """n colors from a dark `base` to a washed-out version, in order.

    For ORDINAL sweeps: lightness encodes the ordering of T, which three
    arbitrary hues do not.
    """
    if n == 1:
        return [mcolors.to_rgb(mcolors.XKCD_COLORS.get(base, base))]
    return [lighten(base, a) for a in np.linspace(lo, hi, n)]


def _np(x):
    """torch tensor (maybe on cuda) / list / ndarray -> float ndarray."""
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    return np.asarray(x, dtype=np.float64)


# The theory curves are evaluated on this dense grid. It is NOT stored in the
# result dicts, so it lives here; keep it in sync with the experiment script.
DEFAULT_TH_NS = np.logspace(np.log10(10), np.log10(1_000_000), num=400)


def _errors_2d(arr, n_ns):
    """(trials, n_ns) regardless of which way round it was stored."""
    a = _np(arr)
    if a.ndim == 1:
        return a[None, :]
    if a.shape[1] == n_ns:
        return a
    if a.shape[0] == n_ns:
        return a.T
    raise ValueError(f"error array {a.shape} matches neither axis to "
                     f"len(emp_ns)={n_ns}")


def _th_ns(data, th_errors, override=None):
    """Resolve the x-grid for the theory curve, and check it lines up.

    Order: explicit override -> a stored th_ns -> DEFAULT_TH_NS -> config['ns']
    / emp_ns if either happens to match by length. Raises rather than plotting
    a theory curve against the wrong x-axis.
    """
    m = len(th_errors)
    if override is not None:
        x = _np(override)
        if len(x) != m:
            raise ValueError(f"th_ns has length {len(x)} but the theory curve "
                             f"has {m} points")
        return x

    r, c = data["results"], data["config"]
    for src in (r, c):
        if src.get("th_ns") is not None and len(_np(src["th_ns"])) == m:
            return _np(src["th_ns"])

    if m == len(DEFAULT_TH_NS):
        return DEFAULT_TH_NS

    for src, key in ((c, "ns"), (r, "emp_ns")):
        if src.get(key) is not None and len(_np(src[key])) == m:
            return _np(src[key])

    raise KeyError(
        f"can't find an x-grid of length {m} for the theory curve. "
        f"DEFAULT_TH_NS has {len(DEFAULT_TH_NS)} points -- if the experiment "
        f"script changed its grid, update DEFAULT_TH_NS or pass th_ns=...")


def _emp_stats(d, key, stderr=True):
    """(emp_ns, mean, err) for one run."""
    r = d["results"]
    emp_ns = _np(r["emp_ns"])
    emp = _errors_2d(r[f"emp_{key}_errors"], len(emp_ns))
    mean, sd = emp.mean(axis=0), emp.std(axis=0)
    if stderr and emp.shape[0] > 1:
        sd = sd / np.sqrt(emp.shape[0])
    return emp_ns, mean, sd


def auto_xlim(runs, xpad=1.8, xtrim=1.15):
    """Span the empirical n values, plus a little breathing room.

    The theory curves run to 1e6 but the markers stop ~2.5e4; showing the full
    theory grid spends most of the panel on a flat asymptote with no data
    under it. Cut a bit past the last marker instead.
    """
    lo = min(_np(d["results"]["emp_ns"]).min() for d in runs.values())
    hi = max(_np(d["results"]["emp_ns"]).max() for d in runs.values())
    return lo / xtrim, hi * xpad


def auto_ylim(runs, keys=("ID",), xlim=None, stderr=True, th_ns=None,
              pad=(0.07, 0.07)):
    """Tight y-limits from only the data inside `xlim`.

    Necessary because matplotlib would autoscale over the whole theory grid,
    including the part beyond the x cutoff -- which is exactly the region
    where the curves bottom out, so it would re-open the gap at the bottom.
    `keys` takes several so ID and OOD panels can share limits.
    """
    xlim = xlim or auto_xlim(runs)
    lo, hi = np.inf, -np.inf
    for key in keys:
        for d in runs.values():
            emp_ns, mean, sd = _emp_stats(d, key, stderr)
            m = (emp_ns >= xlim[0]) & (emp_ns <= xlim[1])
            if m.any():
                lo = min(lo, float((mean - sd)[m].min()))
                hi = max(hi, float((mean + sd)[m].max()))
            th = _np(d["results"][f"th_{key}_errors"]).ravel()
            x = _th_ns(d, th, th_ns)
            m = (x >= xlim[0]) & (x <= xlim[1])
            if m.any():
                lo = min(lo, float(th[m].min()))
                hi = max(hi, float(th[m].max()))
    span = hi - lo
    return lo - pad[0] * span, hi + pad[1] * span


def _corner(ax, text, loc="upper right", fontsize=12):
    x, ha = (0.98, "right") if "right" in loc else (0.02, "left")
    y, va = (0.97, "top") if "upper" in loc else (0.03, "bottom")
    ax.text(x, y, text, ha=ha, va=va, transform=ax.transAxes,
            fontsize=fontsize, color="xkcd:dark gray", linespacing=1.35)


def _legend(ax, title=None, loc="lower left", fontsize=12):
    ax.legend(fontsize=fontsize, title=title, title_fontsize=fontsize,
              framealpha=0.8, handlelength=1, labelspacing=0.15,
              borderpad=0.25, handletextpad=0.4, borderaxespad=0.15, loc=loc)


def config_caption(datas, sweep_key="T"):
    """Corner annotation: the config entries that are the SAME across runs.

    Anything that varies (T) belongs in the legend, not here.
    """
    cfgs = [d["config"] for d in datas]
    c, kp = cfgs[0], cfgs[0].get("kernel_params", {})

    def shared(k, src=lambda c: c):
        v = src(c).get(k)
        return v if all(src(o).get(k) == v for o in cfgs) else None

    bits = []
    kc = shared("kernel_class")
    if kc:
        bits.append(kc.replace("Kernel", " kernel"))
    dims = [f"${k}={shared(k)}$" for k, s in
            (("dim", "d"),) if shared(k) is not None]
    dims = [b.replace("dim", "d") for b in dims]

    # Fix: Correctly append string instead of generator object
    l_val = shared("L")
    if l_val is not None:
        dims.append(f"$L={l_val-1}$")

    ss = shared("sigma_sq")
    if ss is not None:
        dims.append(f"$\\epsilon^2={ss}$")
    if dims:
        bits.append(", ".join(dims))
    kw = shared("kernel_width", lambda c: c.get("kernel_params", {}))
    rg = shared("ridge", lambda c: c.get("kernel_params", {}))
    tail = [s for s in (f"width $={kw}$" if kw is not None else None,
                        f"ridge $={rg}$" if rg is not None else None) if s]
    # if tail:
    #     bits.append(", ".join(tail))
    return "\n".join(bits)


# ------------------------------------------------------- the main workhorse

def add_curves(ax, runs, key="ID", sweep_key="T", base="xkcd:cerulean",
               stderr=True, th_ns=None, colors=None, label_fmt="${v}$",
               fontsize=13):
    """Draw one theory line + one marker cloud per swept value onto `ax`.

    runs  : dict {swept_value: loaded_data}, e.g. {4: loaded_data_4, ...}
    key   : "ID" or "OOD" -- picks emp_{key}_errors / th_{key}_errors
    stderr: divide the trial std by sqrt(trials). Your n=10 bars span ~±0.2
            as raw std; as an uncertainty on the plotted *mean* (20 trials)
            they shrink ~4.5x and stop dominating the left half of the figure.
    """
    vals = sorted(runs)
    datas = [runs[v] for v in vals]
    colors = colors or ramp(len(vals), base=base)

    for v, d, color in zip(vals, datas, colors):
        r = d["results"]
        emp_ns, mean, sd = _emp_stats(d, key, stderr)

        # experiment: markers only, lightened, behind the theory line.
        # ls='none' is the whole trick -- no line through the dots.
        ax.errorbar(emp_ns, mean, yerr=sd, color=lighten(color, 0.45),
                    ls="none", marker="o", ms=4.5, elinewidth=1.2,
                    capsize=0, zorder=-4)

        # theory: the only line in the figure
        th = _np(r[f"th_{key}_errors"]).ravel()
        x_th = _th_ns(d, th, th_ns)
        ax.plot(x_th, th, color=color, lw=2, ls="-",
                label=label_fmt.format(v=v))

    ax.set_xscale("log")
    ax.set_xlabel("$n$ training samples", fontsize=fontsize, labelpad=0)
    return ax


def plot_learning_curves(runs, key="ID", sweep_key="T", base="xkcd:cerulean",
                         stderr=True, th_ns=None, ax=None, figsize=(5.0, 3.0),
                         ylim=None, xlim=None, xpad=1.8, caption=True):
    """Single panel: test MSE vs n, one hue-ramp step per swept value.

    xlim/ylim default to auto: x spans the empirical n plus `xpad`, y is tight
    around whatever falls inside that x range. Pass either to override.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=figsize, constrained_layout=True)

    add_curves(ax, runs, key=key, sweep_key=sweep_key, base=base,
               stderr=stderr, th_ns=th_ns)

    xlim = xlim or auto_xlim(runs, xpad=xpad)
    ylim = ylim or auto_ylim(runs, keys=(key,), xlim=xlim, stderr=stderr,
                             th_ns=th_ns)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_ylabel("Test MSE", fontsize=13)
    _legend(ax, title=f"${sweep_key}$", loc="lower left")
    if caption:
        cap = config_caption([runs[v] for v in sorted(runs)], sweep_key)
        _corner(ax, f"{key}\n{cap}")
    return ax


def plot_id_vs_ood(runs, sweep_key="T", base="xkcd:cerulean", stderr=True,
                   th_ns=None, figsize=(8.4, 3.0), ylim=None, xlim=None,
                   xpad=1.8, caption_loc="lower left", fontsize=13):
    """Two panels sharing a y-axis: ID | OOD.

    Small multiples rather than 6 curves in one axes -- the papers never go
    past ~7 curves per panel, and ID/OOD on shared limits is easier to compare
    than solid-vs-dashed in a single crowded panel. The shared y-limits are
    computed over both panels so the comparison stays honest.
    """
    fig, (ax_id, ax_ood) = plt.subplots(
        1, 2, figsize=figsize, sharey=True, constrained_layout=True,
        gridspec_kw=dict(wspace=0.05))

    xlim = xlim or auto_xlim(runs, xpad=xpad)#Pretrain Distribution #True Distribution
    ylim = ylim or auto_ylim(runs, keys=("ID", "OOD"), xlim=xlim,
                             stderr=stderr, th_ns=th_ns)

    for ax, key in ((ax_id, "ID"), (ax_ood, "OOD")):
        add_curves(ax, runs, key=key, sweep_key=sweep_key, base=base,
                   stderr=stderr, th_ns=th_ns, fontsize=fontsize)
        k = "Pretrain Distribution" if key == "ID" else "True Distribution"
        _corner(ax, k, fontsize=fontsize)
        ax.set_xlim(*xlim)

    ax_id.set_ylabel("Test MSE", fontsize=fontsize)
    ax_id.set_ylim(*ylim)
    _legend(ax_id, title=f"${sweep_key}$", loc="lower left", fontsize=fontsize)
    _corner(ax_ood, config_caption([runs[v] for v in sorted(runs)], sweep_key),
            loc=caption_loc, fontsize=fontsize)
    return fig, (ax_id, ax_ood)


# --------------------------------------- escape hatch: collapse each curve

def sample_complexity(ns, curve, threshold=1.0):
    """n at which a decreasing curve crosses `threshold`, interpolated in log n."""
    ns, curve = _np(ns), _np(curve)
    return 10 ** np.interp(threshold, curve[::-1], np.log10(ns)[::-1])


def plot_sample_complexities(runs, key="ID", threshold=1.0, sweep_key="T",
                             base="xkcd:cerulean", th_ns=None, ax=None,
                             figsize=(3.6, 3.6), lims=None):
    """Collapse each curve to one point: empirical vs predicted n_lrn, vs y=x.

    Use when the curves overlap too much to read. This is how Fig 3 (top right)
    of the HEA paper fits dozens of targets into one panel.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=figsize, constrained_layout=True)
    vals = sorted(runs)
    colors = ramp(len(vals), base=base)
    pts = []
    for v, color in zip(vals, colors):
        d = runs[v]
        r = d["results"]
        emp_ns = _np(r["emp_ns"])
        emp = _errors_2d(r[f"emp_{key}_errors"], len(emp_ns)).mean(axis=0)
        th = _np(r[f"th_{key}_errors"]).ravel()
        x_th = _th_ns(d, th, th_ns)
        n_emp = sample_complexity(emp_ns, emp, threshold)
        n_th = sample_complexity(x_th, th, threshold)
        pts += [n_emp, n_th]
        ax.scatter(n_emp, n_th, color=color, lw=0, zorder=1, s=42,
                   label=f"${v}$")
    lims = lims or (0.5 * min(pts), 2 * max(pts))
    ax.plot(lims, lims, color="gray", alpha=0.5, zorder=-4)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlim(*lims); ax.set_ylim(*lims)
    ax.set_xlabel("empirical $n_\\mathrm{lrn}$", fontsize=13)
    ax.set_ylabel("predicted $n_\\mathrm{lrn}$", fontsize=13)
    _legend(ax, title=f"${sweep_key}$", loc="lower right")
    return ax

In [ ]:
from pathlib import Path

outdir = Path("/content/drive/MyDrive/kernel_icl/figs")
# outdir.mkdir(parents=True, exist_ok=True)


In [ ]:
runs = {1: loaded_data_1, 4: loaded_data_4, 16: loaded_data_16, 256: loaded_data_256}

# plot_learning_curves(runs)              # ID only, single panel
fig, _ = plot_id_vs_ood(runs, stderr=False, fontsize=16)                    # ID | OOD, shared y-axis
# fig.savefig("KICL_learning_curves_d5_L8.pdf")
fig.savefig(outdir / "KICL_learning_curves_d5_L7.pdf")
# plot_sample_complexities(runs, threshold=1.0)   # if it still gets crowded